# Object-Oriented Programming (OOP) in Python — Problems and Solutions
**Technical Notes**  
*Rodrigo Kang*

This notebook contains complete solutions to the problems from the associated OOP technical notes. The emphasis is on reasoning about object state, behaviour, interfaces, inheritance, composition, polymorphism, and the stateful design patterns that appear frequently in data science and machine learning code.

In [ ]:
import math
import numpy as np

from dataclasses import dataclass, field
from fractions import Fraction

## Problem 1 — RationalNumber

### Problem

Create a `RationalNumber` class with integer `numerator` and `denominator` attributes. The denominator defaults to `1`, both arguments must be integers, and the denominator cannot be zero.

### Solution

The constructor is responsible for establishing valid object state. Type and denominator validation should occur before assigning the attributes so that an invalid rational number is never created successfully.

#### Implementation

In [ ]:
class RationalNumber:
    def __init__(self, numerator, denominator=1):
        if not isinstance(numerator, int) or not isinstance(denominator, int):
            raise TypeError("Numerator and denominator must be integers.")
        if denominator == 0:
            raise ValueError("Denominator cannot be zero.")

        self.numerator = numerator
        self.denominator = denominator


r1 = RationalNumber(3, 4)
r2 = RationalNumber(5)

print(r1.__dict__)
print(r2.__dict__)

### Key Takeaway

Each instance owns its own numerator and denominator. The class defines the common rules; the instances hold independent state.

## Problem 2 — String Representation

### Problem

Extend `RationalNumber` with `__str__()` and `__repr__()`.

### Solution

`__str__()` should favour readable user-facing output, while `__repr__()` should make the object's type and important state explicit for debugging and inspection.

#### Implementation

In [ ]:
class RationalNumber:
    def __init__(self, numerator, denominator=1):
        if not isinstance(numerator, int) or not isinstance(denominator, int):
            raise TypeError("Numerator and denominator must be integers.")
        if denominator == 0:
            raise ValueError("Denominator cannot be zero.")
        self.numerator = numerator
        self.denominator = denominator

    def __str__(self):
        return f"{self.numerator} / {self.denominator}"

    def __repr__(self):
        return (
            f"RationalNumber("
            f"numerator={self.numerator!r}, "
            f"denominator={self.denominator!r})"
        )


r = RationalNumber(3, 4)
print(str(r))
print(repr(r))

### Key Takeaway

Special methods let user-defined classes participate naturally in ordinary Python operations such as `str()`, `repr()`, and `print()`.

## Problem 3 — Rational Number Behaviour

### Problem

Add a computed `quotient` property and a `simplify()` method.

### Solution

Here `simplify()` mutates the existing object. The greatest common divisor is used to divide numerator and denominator by their common factor; the denominator is normalised to remain positive.

#### Implementation

In [ ]:
class RationalNumber:
    def __init__(self, numerator, denominator=1):
        if not isinstance(numerator, int) or not isinstance(denominator, int):
            raise TypeError("Numerator and denominator must be integers.")
        if denominator == 0:
            raise ValueError("Denominator cannot be zero.")
        self.numerator = numerator
        self.denominator = denominator
        self._normalise_sign()

    def _normalise_sign(self):
        if self.denominator < 0:
            self.numerator *= -1
            self.denominator *= -1

    @property
    def quotient(self):
        return self.numerator / self.denominator

    def simplify(self):
        divisor = math.gcd(self.numerator, self.denominator)
        self.numerator //= divisor
        self.denominator //= divisor
        self._normalise_sign()
        return self

    def __str__(self):
        return f"{self.numerator} / {self.denominator}"


q = RationalNumber(8, 12)
print("Before:", q, q.quotient)
q.simplify()
print("After:", q, q.quotient)

### Key Takeaway

Mutation is acceptable when it is explicit and semantically natural. Returning `self` also makes fluent use possible.

## Problem 4 — Static Methods for Rational Arithmetic

### Problem

Add static methods for addition, subtraction, multiplication, and division of two rational numbers.

### Solution

These operations need two explicit rational-number objects but neither the current instance nor class-level state, so static methods are a defensible design. Operator overloading would make the interface more idiomatic for arithmetic.

#### Implementation

In [ ]:
class RationalNumber:
    def __init__(self, numerator, denominator=1):
        if not isinstance(numerator, int) or not isinstance(denominator, int):
            raise TypeError("Numerator and denominator must be integers.")
        if denominator == 0:
            raise ValueError("Denominator cannot be zero.")
        self.numerator = numerator
        self.denominator = denominator

    def simplify(self):
        divisor = math.gcd(self.numerator, self.denominator)
        self.numerator //= divisor
        self.denominator //= divisor
        if self.denominator < 0:
            self.numerator *= -1
            self.denominator *= -1
        return self

    @staticmethod
    def add(p, q):
        return RationalNumber(
            p.numerator * q.denominator + q.numerator * p.denominator,
            p.denominator * q.denominator
        ).simplify()

    @staticmethod
    def subtract(p, q):
        return RationalNumber(
            p.numerator * q.denominator - q.numerator * p.denominator,
            p.denominator * q.denominator
        ).simplify()

    @staticmethod
    def multiply(p, q):
        return RationalNumber(
            p.numerator * q.numerator,
            p.denominator * q.denominator
        ).simplify()

    @staticmethod
    def divide(p, q):
        if q.numerator == 0:
            raise ZeroDivisionError("Cannot divide by zero.")
        return RationalNumber(
            p.numerator * q.denominator,
            p.denominator * q.numerator
        ).simplify()

    def __str__(self):
        return f"{self.numerator} / {self.denominator}"


p = RationalNumber(1, 2)
q = RationalNumber(2, 3)

print(RationalNumber.add(p, q))
print(RationalNumber.subtract(p, q))
print(RationalNumber.multiply(p, q))
print(RationalNumber.divide(p, q))

### Key Takeaway

A static method belongs to the class namespace but does not receive `self` or `cls` automatically.

## Problem 5 — Alternative Constructors

### Problem

Add `zero()`, `one()`, and `from_float()` class methods.

### Solution

Class methods are appropriate because they create new instances through `cls`. Using `Fraction(str(value))` gives a rational representation of the decimal text rather than the exact binary floating-point value.

#### Implementation

In [ ]:
class RationalNumber:
    def __init__(self, numerator, denominator=1):
        if not isinstance(numerator, int) or not isinstance(denominator, int):
            raise TypeError("Numerator and denominator must be integers.")
        if denominator == 0:
            raise ValueError("Denominator cannot be zero.")
        self.numerator = numerator
        self.denominator = denominator

    @classmethod
    def zero(cls):
        return cls(0, 1)

    @classmethod
    def one(cls):
        return cls(1, 1)

    @classmethod
    def from_float(cls, value):
        if not isinstance(value, (int, float)):
            raise TypeError("Value must be numeric.")
        fraction = Fraction(str(value))
        return cls(fraction.numerator, fraction.denominator)

    def __str__(self):
        return f"{self.numerator} / {self.denominator}"


print(RationalNumber.zero())
print(RationalNumber.one())
print(RationalNumber.from_float(5.4))

### Key Takeaway

Alternative constructors build the same type from different input representations while remaining subclass-friendly through `cls`.

## Problem 6 — Properties in `RationalNumber`

### Problem

Expose `quotient` and `is_proper` as read-only properties.

### Solution

Both values are fully determined by existing state, so storing them separately would introduce redundant state that could become inconsistent.

#### Implementation

In [ ]:
class RationalNumber:
    def __init__(self, numerator, denominator=1):
        if not isinstance(numerator, int) or not isinstance(denominator, int):
            raise TypeError("Numerator and denominator must be integers.")
        if denominator == 0:
            raise ValueError("Denominator cannot be zero.")
        self.numerator = numerator
        self.denominator = denominator

    @property
    def quotient(self):
        return self.numerator / self.denominator

    @property
    def is_proper(self):
        return abs(self.numerator) < abs(self.denominator)


q1 = RationalNumber(3, 4)
q2 = RationalNumber(5, 2)

print(q1.quotient, q1.is_proper)
print(q2.quotient, q2.is_proper)

### Key Takeaway

Properties are well suited to derived values that conceptually behave like attributes.

## Problem 7 — `Point3D` from `Point2D`

### Problem

Create `Point3D` as a subclass of the supplied `Point2D`, adding `z` and overriding the required behaviour.

### Solution

The 3D point reuses the parent initialiser for `x` and `y`, then adds `z`. It overrides the string representation and the alternative `zero()` constructor because those behaviours depend on dimensionality.

#### Implementation

In [ ]:
class Point2D:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __str__(self):
        return f"({self.x}, {self.y})"

    @classmethod
    def zero(cls):
        return cls(0, 0)


class Point3D(Point2D):
    def __init__(self, x, y, z):
        super().__init__(x, y)
        self.z = z

    def __str__(self):
        return f"({self.x}, {self.y}, {self.z})"

    @classmethod
    def zero(cls):
        return cls(0, 0, 0)


p = Point3D(1, 2, 3)
z = Point3D.zero()
print(p)
print(z)
print(isinstance(p, Point2D))

### Key Takeaway

Inheritance reuses the common 2D state while overriding only behaviour that must change in three dimensions.

## Problem 8 — Multiple Inheritance and `Pyramid`

### Problem

Implement a regular square pyramid and inspect the design implications of multiple inheritance.

### Solution

A pyramid is not naturally both a square and a triangle as an object identity, so composition or a direct class is clearer than multiple inheritance. The implementation below uses a direct `Pyramid` class while still demonstrating the relevant formulas.

#### Implementation

In [ ]:
class Pyramid:
    def __init__(self, base, height):
        if base <= 0 or height <= 0:
            raise ValueError("Base and height must be positive.")
        self.base = base
        self.height = height

    @property
    def slant_height(self):
        return math.sqrt((self.base / 2) ** 2 + self.height ** 2)

    @property
    def area(self):
        return self.base ** 2 + 2 * self.base * self.slant_height

    @property
    def volume(self):
        return (self.base ** 2 * self.height) / 3


pyramid = Pyramid(base=4, height=3)
print("slant height:", pyramid.slant_height)
print("surface area:", pyramid.area)
print("volume:", pyramid.volume)

### Key Takeaway

Inheritance should model a genuine type relationship. Reuse alone is not sufficient justification for multiple inheritance.

## Problem 9 — `Student`

### Problem

Define `Student` with `sid`, `name`, `gender`, `type="learning"`, and a `say_name()` instance method.

### Solution

The method receives the instance automatically through `self` when called with dot notation.

#### Implementation

In [ ]:
class Student:
    def __init__(self, sid, name, gender, type="learning"):
        self.sid = sid
        self.name = name
        self.gender = gender
        self.type = type

    def say_name(self):
        return f"My name is {self.name}."


student1 = Student("001", "Susan", "F")
student2 = Student("002", "Mike", "M")

print(student1.say_name())
print(student2.say_name())

### Key Takeaway

`student.say_name()` is effectively a bound-method call in which Python supplies `student` as `self`.

## Problem 10 — Methods with Additional Arguments

### Problem

Extend `Student` with a validated `report(score)` method.

### Solution

`self` identifies which student is reporting; `score` is call-specific input. The method validates the score before formatting the result.

#### Implementation

In [ ]:
class Student:
    def __init__(self, sid, name, gender, type="learning"):
        self.sid = sid
        self.name = name
        self.gender = gender
        self.type = type

    def report(self, score):
        if not 0 <= score <= 100:
            raise ValueError("Score must be between 0 and 100.")
        return f"{self.name} ({self.sid}): {score}"


student = Student("001", "Susan", "F")
print(student.report(88))

### Key Takeaway

Instance state and call-specific parameters play different roles inside a method.

## Problem 11 — Instance Independence

### Problem

Modify one student's `type` without modifying another student's `type`.

### Solution

Instance attributes live on individual objects. Assigning `student1.type` creates or replaces state on `student1` only.

#### Implementation

In [ ]:
student1 = Student("001", "Susan", "F")
student2 = Student("002", "Mike", "M")

student1.type = "researching"

print(student1.__dict__)
print(student2.__dict__)

### Key Takeaway

Objects created from the same class share the class definition, not their ordinary instance state.

## Problem 12 — Counting Instances

### Problem

Track the number of `Student` instances with a class attribute and class method.

### Solution

The count describes the class population rather than one particular student, so it belongs at class level.

#### Implementation

In [ ]:
class Student:
    n = 0

    def __init__(self, sid, name, gender, type="learning"):
        self.sid = sid
        self.name = name
        self.gender = gender
        self.type = type
        self.__class__.n += 1

    @classmethod
    def num_instances(cls):
        return cls.n


s1 = Student("001", "Susan", "F")
s2 = Student("002", "Mike", "M")
s3 = Student("003", "Ana", "F")

print(Student.num_instances())

### Key Takeaway

Class attributes represent state associated with the class as a whole rather than any one instance.

## Problem 13 — `Sensor`

### Problem

Create a sensor with `name`, `location`, `record_date`, and per-instance `data`; implement `add_data()` and `clear_data()`.

### Solution

The mutable dictionary must be created inside `__init__()` so each sensor owns independent storage. Input arrays are validated for equal length.

#### Implementation

In [ ]:
class Sensor:
    def __init__(self, name, location, record_date):
        self.name = name
        self.location = location
        self.record_date = record_date
        self.data = {}

    def add_data(self, t, data):
        if len(t) != len(data):
            raise ValueError("Time and data must have the same length.")
        self.data["time"] = np.asarray(t)
        self.data["data"] = np.asarray(data)
        return self

    def clear_data(self):
        self.data.clear()


sensor = Sensor("S1", "Lab", "2026-08-18")
sensor.add_data(
    t=np.array([0, 1, 2]),
    data=np.array([10.0, 10.5, 11.1])
)
print(sensor.data)
sensor.clear_data()
print(sensor.data)

### Key Takeaway

Mutable defaults belong on the instance unless sharing them across every object is explicitly intended.

## Problem 14 — Overriding with `Accelerometer`

### Problem

Create `Accelerometer` and `UCBAcc`, overriding `show_type()` and extending it through `super()`.

### Solution

Inheritance gives the subclasses the sensor state. `Accelerometer` adds a type description, and `UCBAcc` overrides that method while reusing the parent implementation.

#### Implementation

In [ ]:
class Sensor:
    def __init__(self, name, location, record_date):
        self.name = name
        self.location = location
        self.record_date = record_date
        self.data = {}


class Accelerometer(Sensor):
    def show_type(self):
        return "I am an accelerometer."


class UCBAcc(Accelerometer):
    def show_type(self):
        return f"{super().show_type()} My name is {self.name}."


acc = UCBAcc("UCB-01", "Berkeley", "2026-08-18")
print(acc.show_type())

### Key Takeaway

Overriding replaces inherited behaviour for a subclass; `super()` can reuse rather than duplicate the parent implementation.

## Problem 15 — Extending a Parent Initialiser

### Problem

Create `NewSensor` with an additional `brand` attribute and call `super().__init__()`.

### Solution

The parent remains the single source of truth for common sensor initialisation. If the parent later changes, the subclass automatically reuses the updated initialisation logic.

#### Implementation

In [ ]:
class NewSensor(Sensor):
    def __init__(self, name, location, record_date, brand):
        super().__init__(name, location, record_date)
        self.brand = brand


new_sensor = NewSensor(
    name="S2",
    location="Field",
    record_date="2026-08-18",
    brand="Acme"
)
print(new_sensor.__dict__)

### Key Takeaway

`super()` reduces duplication and keeps subclass behaviour coupled to the intended parent interface rather than its implementation details.

## Problem 16 — `RegularPolygon`

### Problem

Create a validated regular-polygon class with computed apothem, perimeter, and area.

### Solution

The polygon stores only the independent state (`base` and `n`). Geometric quantities are calculated from that state to avoid inconsistency.

#### Implementation

In [ ]:
class RegularPolygon:
    def __init__(self, base, n):
        if base <= 0:
            raise ValueError("Base must be positive.")
        if n < 3:
            raise ValueError("A polygon must have at least three sides.")
        self.base = base
        self.n = n

    @property
    def apothem(self):
        return self.base / (2 * math.tan(math.pi / self.n))

    @property
    def perimeter(self):
        return self.n * self.base

    @property
    def area(self):
        return self.perimeter * self.apothem / 2

    def __str__(self):
        return f"RegularPolygon(base={self.base}, n={self.n})"


polygon = RegularPolygon(base=4, n=5)
print(polygon)
print(polygon.apothem, polygon.perimeter, polygon.area)

### Key Takeaway

Store independent state; compute quantities that are deterministic functions of that state.

## Problem 17 — Polygon Subclasses

### Problem

Create `Triangle`, `Square`, and `Pentagon` as specialisations of `RegularPolygon`.

### Solution

Each subclass fixes the number of sides while reusing all geometric behaviour from the parent.

#### Implementation

In [ ]:
class Triangle(RegularPolygon):
    def __init__(self, base):
        super().__init__(base=base, n=3)


class Square(RegularPolygon):
    def __init__(self, base):
        super().__init__(base=base, n=4)


class Pentagon(RegularPolygon):
    def __init__(self, base):
        super().__init__(base=base, n=5)


triangle = Triangle(4)
square = Square(4)
pentagon = Pentagon(4)

print(triangle.area)
print(square.area)
print(pentagon.area)

### Key Takeaway

This is a natural `is-a` relationship: each specialised shape is a regular polygon with a fixed side count.

## Problem 18 — Three-Dimensional Shapes

### Problem

Create `Tetrahedron` and `Cube` using the polygon hierarchy and add surface-area and volume properties.

### Solution

The 3D objects reuse side-length state and some planar geometry, while adding volume and total surface-area semantics.

#### Implementation

In [ ]:
class Tetrahedron(Triangle):
    @property
    def surface_area(self):
        return math.sqrt(3) * self.base ** 2

    @property
    def volume(self):
        return self.base ** 3 / (6 * math.sqrt(2))


class Cube(Square):
    @property
    def surface_area(self):
        return 6 * self.base ** 2

    @property
    def volume(self):
        return self.base ** 3


tetra = Tetrahedron(3)
cube = Cube(3)

print(tetra.surface_area, tetra.volume)
print(cube.surface_area, cube.volume)

### Key Takeaway

Inheritance can reuse common side-length structure, but inherited properties should only be exposed when their interpretation remains meaningful.

## Problem 19 — `Circle`

### Problem

Create a validated `Circle` with computed diameter, perimeter, and area.

### Solution

Only radius needs to be stored. The other quantities are deterministic functions of radius.

#### Implementation

In [ ]:
class Circle:
    def __init__(self, radius):
        if radius <= 0:
            raise ValueError("Radius must be positive.")
        self.radius = radius

    @property
    def diameter(self):
        return 2 * self.radius

    @property
    def perimeter(self):
        return 2 * math.pi * self.radius

    @property
    def area(self):
        return math.pi * self.radius ** 2


circle = Circle(3)
print(circle.diameter, circle.perimeter, circle.area)

### Key Takeaway

Computed properties avoid redundant state and automatically stay consistent when the underlying radius changes.

## Problem 20 — `Cylinder`: Inheritance or Composition?

### Problem

Implement a cylinder and justify inheritance or composition.

### Solution

A cylinder is not literally a circle, but it has circular bases. Composition therefore models the relationship more accurately.

#### Implementation

In [ ]:
class Circle:
    def __init__(self, radius):
        if radius <= 0:
            raise ValueError("Radius must be positive.")
        self.radius = radius

    @property
    def area(self):
        return math.pi * self.radius ** 2

    @property
    def perimeter(self):
        return 2 * math.pi * self.radius


class Cylinder:
    def __init__(self, radius, height):
        if height <= 0:
            raise ValueError("Height must be positive.")
        self.base = Circle(radius)
        self.height = height

    @property
    def surface_area(self):
        return 2 * self.base.area + self.base.perimeter * self.height

    @property
    def volume(self):
        return self.base.area * self.height


cylinder = Cylinder(radius=2, height=5)
print(cylinder.surface_area, cylinder.volume)

### Key Takeaway

Use composition for `has-a` relationships and inheritance for genuine `is-a` relationships.

## Problem 21 — Identity Versus Equality

### Problem

Compare two rectangles before and after defining `__eq__()`.

### Solution

`is` always tests whether two references identify the same object. `==` can be given domain-specific value semantics with `__eq__()`.

#### Implementation

In [ ]:
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height


a = Rectangle(4, 3)
b = Rectangle(4, 3)

print("Before __eq__:", a is b, a == b)


class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def __eq__(self, other):
        if not isinstance(other, Rectangle):
            return NotImplemented
        return self.width == other.width and self.height == other.height


a = Rectangle(4, 3)
b = Rectangle(4, 3)

print("After __eq__:", a is b, a == b)

### Key Takeaway

Identity is about object references; equality is about a class-defined notion of equivalent value.

## Problem 22 — A Read-Only Property

### Problem

Implement a bank account whose balance can only change through validated deposit and withdrawal methods.

### Solution

The public `balance` property exposes state for reading while the mutation methods protect the invariants.

#### Implementation

In [ ]:
class BankAccount:
    def __init__(self, initial_balance=0):
        if initial_balance < 0:
            raise ValueError("Initial balance cannot be negative.")
        self._balance = initial_balance

    @property
    def balance(self):
        return self._balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Deposit must be positive.")
        self._balance += amount
        return self

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError("Withdrawal must be positive.")
        if amount > self._balance:
            raise ValueError("Insufficient funds.")
        self._balance -= amount
        return self


account = BankAccount(100)
account.deposit(50).withdraw(20)
print(account.balance)

### Key Takeaway

Encapsulation is mainly about maintaining a clear, valid public interface around object state.

## Problem 23 — Class, Static, or Instance Method?

### Problem

Classify five operations as instance method, class method, static method, or ordinary function.

### Solution

A reasonable design is: rectangle area → instance method/property; construct from `"10x5"` → class method; compare rectangle dimensions → static method or ordinary function; Euclidean distance between arbitrary tuples → ordinary function; count students → class method reading a class attribute.

### Key Takeaway

Choose the method category according to the context the operation actually needs: instance, class, or neither.

## Problem 24 — Duck Typing

### Problem

Create unrelated `Dog` and `Robot` classes that both implement `speak()`.

### Solution

The calling function does not need inheritance or explicit type tests; it only requires the expected behaviour.

#### Implementation

In [ ]:
class Dog:
    def speak(self):
        return "Woof"


class Robot:
    def speak(self):
        return "Synthetic voice"


def make_speak(obj):
    return obj.speak()


print(make_speak(Dog()))
print(make_speak(Robot()))

### Key Takeaway

Duck typing makes interfaces behavioural: if an object supports the operation, client code can often use it.

## Problem 25 — Polymorphic Interface

### Problem

Create three simple model classes supporting `fit(X, y)` and `predict(X)` and use them through one loop.

### Solution

The models expose a common interface even though their internal fitted state differs. The loop depends on behaviour rather than exact class identity.

#### Implementation

In [ ]:
class LinearModel:
    def fit(self, X, y):
        X_aug = np.column_stack([np.ones(len(X)), X])
        self.coef_ = np.linalg.pinv(X_aug) @ y
        return self

    def predict(self, X):
        X_aug = np.column_stack([np.ones(len(X)), X])
        return X_aug @ self.coef_


class TreeModel:
    def fit(self, X, y):
        self.threshold_ = float(np.median(X[:, 0]))
        self.left_ = float(np.mean(y[X[:, 0] <= self.threshold_]))
        self.right_ = float(np.mean(y[X[:, 0] > self.threshold_]))
        return self

    def predict(self, X):
        return np.where(
            X[:, 0] <= self.threshold_,
            self.left_,
            self.right_
        )


class MeanModel:
    def fit(self, X, y):
        self.mean_ = float(np.mean(y))
        return self

    def predict(self, X):
        return np.full(len(X), self.mean_)


X25 = np.array([[0.0], [1.0], [2.0], [3.0]])
y25 = np.array([0.0, 1.0, 1.8, 3.2])

models25 = [LinearModel(), TreeModel(), MeanModel()]

for model in models25:
    model.fit(X25, y25)
    print(type(model).__name__, model.predict(X25))

### Key Takeaway

Polymorphism lets client code depend on a stable interface while implementations vary behind it.

## Problem 26 — Method Resolution Order

### Problem

Predict and inspect method resolution for classes `C(A, B)` and `D(B, A)`.

### Solution

Python searches according to the MRO. In these simple cases, `C` reaches `A` before `B`, while `D` reaches `B` before `A`.

#### Implementation

In [ ]:
class A:
    def identify(self):
        return "A"


class B:
    def identify(self):
        return "B"


class C(A, B):
    pass


class D(B, A):
    pass


print(C().identify())
print(C.mro())
print(D().identify())
print(D.mro())

### Key Takeaway

The MRO is the general lookup rule; parent-list order is only one input into that algorithm.

## Problem 27 — Cooperative `super()`

### Problem

Construct an inheritance chain in which every `process()` method contributes through `super()`.

### Solution

Each override calls the next method in the MRO rather than naming a parent directly. This keeps the hierarchy cooperative.

#### Implementation

In [ ]:
class A:
    def process(self):
        return ["A"]


class B(A):
    def process(self):
        result = super().process()
        result.append("B")
        return result


class C(B):
    def process(self):
        result = super().process()
        result.append("C")
        return result


print(C().process())
print(C.mro())

### Key Takeaway

`super()` supports cooperative method chains because it follows the current MRO rather than hard-coding a parent class.

## Problem 28 — Composition

### Problem

Create `Scaler`, `Model`, and `Pipeline` such that the pipeline has both components and coordinates fitting and prediction.

### Solution

The pipeline is not a scaler or a model. It owns those components and delegates work to them, which is a natural composition design.

#### Implementation

In [ ]:
class Scaler:
    def fit(self, X):
        self.mean_ = X.mean(axis=0)
        self.std_ = X.std(axis=0)
        if np.any(self.std_ == 0):
            raise ValueError("Zero-variance feature.")
        return self

    def transform(self, X):
        return (X - self.mean_) / self.std_


class Model:
    def fit(self, X, y):
        X_aug = np.column_stack([np.ones(len(X)), X])
        self.coef_ = np.linalg.pinv(X_aug) @ y
        return self

    def predict(self, X):
        X_aug = np.column_stack([np.ones(len(X)), X])
        return X_aug @ self.coef_


class Pipeline:
    def __init__(self, scaler, model):
        self.scaler = scaler
        self.model = model

    def fit(self, X, y):
        X_scaled = self.scaler.fit(X).transform(X)
        self.model.fit(X_scaled, y)
        return self

    def predict(self, X):
        return self.model.predict(self.scaler.transform(X))


X28 = np.array([[1.0, 10.0], [2.0, 20.0], [3.0, 30.0], [4.0, 40.0]])
y28 = np.array([1.0, 2.0, 3.0, 4.0])

pipe28 = Pipeline(Scaler(), Model()).fit(X28, y28)
print(pipe28.predict(X28))

### Key Takeaway

Composition makes replaceable components explicit and avoids artificial multiple-inheritance relationships.

## Problem 29 — Dataclass

### Problem

Implement `ModelResult` with `@dataclass`, test equality, and compare with an ordinary class.

### Solution

The dataclass decorator generates common machinery such as `__init__()`, `__repr__()`, and value-based `__eq__()` from the declared fields.

#### Implementation

In [ ]:
@dataclass
class ModelResult:
    model_name: str
    rmse: float
    training_time: float


r1 = ModelResult("Linear", 3.2, 1.1)
r2 = ModelResult("Linear", 3.2, 1.1)

print(r1)
print(r1 == r2)


class ModelResultManual:
    def __init__(self, model_name, rmse, training_time):
        self.model_name = model_name
        self.rmse = rmse
        self.training_time = training_time

    def __repr__(self):
        return (
            f"ModelResultManual("
            f"model_name={self.model_name!r}, "
            f"rmse={self.rmse!r}, "
            f"training_time={self.training_time!r})"
        )

    def __eq__(self, other):
        if not isinstance(other, ModelResultManual):
            return NotImplemented
        return (
            self.model_name == other.model_name
            and self.rmse == other.rmse
            and self.training_time == other.training_time
        )

### Key Takeaway

Dataclasses are ordinary classes with generated boilerplate, especially useful for data-oriented objects.

## Problem 30 — Mutable Defaults in Dataclasses

### Problem

Use `field(default_factory=list)` so each experiment receives its own metrics list.

### Solution

A default factory is evaluated for each new instance, preventing accidental sharing of one mutable object.

#### Implementation

In [ ]:
@dataclass
class Experiment:
    metrics: list = field(default_factory=list)


e1 = Experiment()
e2 = Experiment()

e1.metrics.append(0.9)

print(e1.metrics)
print(e2.metrics)
print(e1.metrics is e2.metrics)

### Key Takeaway

Mutable per-instance state should be created per instance, not shared as a class-level default object.

## Problem 31 — `Dataset` Abstraction

### Problem

Create a dataset object storing `name`, `X`, and `y`, validating sample counts and reporting dimensions.

### Solution

The class keeps related arrays and metadata together and validates an invariant at construction time.

#### Implementation

In [ ]:
class Dataset:
    def __init__(self, name, X, y):
        X = np.asarray(X)
        y = np.asarray(y)

        if len(X) != len(y):
            raise ValueError("X and y must contain the same number of observations.")
        if X.ndim != 2:
            raise ValueError("X must be a two-dimensional feature matrix.")

        self.name = name
        self.X = X
        self.y = y

    @property
    def n_samples(self):
        return self.X.shape[0]

    @property
    def n_features(self):
        return self.X.shape[1]

    def __repr__(self):
        return (
            f"Dataset(name={self.name!r}, "
            f"n_samples={self.n_samples}, "
            f"n_features={self.n_features})"
        )


dataset31 = Dataset(
    "example",
    X=np.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]]),
    y=np.array([0, 1, 0])
)
dataset31

### Key Takeaway

A useful class packages state, invariants, and behaviour that belong to one coherent abstraction.

## Problem 32 — Stateful `Standardizer`

### Problem

Implement a transformer that learns `mean_` and `std_`, validates fit state, and rejects zero-variance features.

### Solution

The learned statistics persist after `fit()` so that exactly the same transformation can later be applied to new observations.

#### Implementation

In [ ]:
class Standardizer:
    def __init__(self):
        self.mean_ = None
        self.std_ = None

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        self.mean_ = X.mean(axis=0)
        self.std_ = X.std(axis=0)

        if np.any(self.std_ == 0):
            self.mean_ = None
            self.std_ = None
            raise ValueError("All features must have non-zero standard deviation.")

        return self

    def transform(self, X):
        if self.mean_ is None or self.std_ is None:
            raise RuntimeError("The transformer must be fitted first.")
        X = np.asarray(X, dtype=float)
        return (X - self.mean_) / self.std_

    def fit_transform(self, X):
        return self.fit(X).transform(X)


X32 = np.array([
    [1.0, 10.0],
    [2.0, 20.0],
    [3.0, 30.0],
    [4.0, 40.0]
])

standardizer32 = Standardizer()
Z32 = standardizer32.fit_transform(X32)

print(standardizer32.mean_)
print(standardizer32.std_)
print(Z32)

### Key Takeaway

Persistent learned state is what makes estimator-style OOP interfaces natural in machine learning.

## Problem 33 — Separate Fit and Transform Data

### Problem

Fit the standardizer on training data only and apply it unchanged to train and test data.

### Solution

The training set determines the learned transformation. Re-fitting on the test set would make the test representation depend on test-distribution information and would invalidate the evaluation boundary.

#### Implementation

In [ ]:
X_train33 = np.array([
    [1.0, 10.0],
    [2.0, 20.0],
    [3.0, 30.0],
    [4.0, 40.0]
])

X_test33 = np.array([
    [5.0, 50.0],
    [6.0, 60.0]
])

standardizer33 = Standardizer().fit(X_train33)

train_scaled33 = standardizer33.transform(X_train33)
test_scaled33 = standardizer33.transform(X_test33)

print(train_scaled33)
print(test_scaled33)

### Key Takeaway

Fitted preprocessing objects must preserve training-derived state and reuse it consistently on validation, test, and future data.

## Problem 34 — Alternative Constructor for a Model Configuration

### Problem

Create a `ModelConfig` dataclass with a `from_dict()` class method.

### Solution

The class method converts an external mapping representation into an instance of the class and remains compatible with subclasses through `cls`.

#### Implementation

In [ ]:
@dataclass
class ModelConfig:
    learning_rate: float = 0.1
    max_depth: int = 3
    random_state: int = 42

    @classmethod
    def from_dict(cls, values):
        return cls(**values)


config34 = ModelConfig.from_dict({
    "learning_rate": 0.05,
    "max_depth": 5,
    "random_state": 7
})

config34

### Key Takeaway

Class methods are natural alternative constructors when the class itself is needed to build the returned object.

## Problem 35 — Inheritance or Composition?

### Problem

Choose inheritance, composition, or neither for six relationships.

### Solution

A defensible classification is: Employee–Person → inheritance; Car–Engine → composition; Pipeline–Estimator → composition; Square–RegularPolygon → inheritance; Dataset–DataFrame → usually neither (Dataset may contain or wrap a DataFrame if useful); RandomForestModel–generic estimator interface → interface inheritance or protocol can be appropriate, though duck typing may also suffice.

### Key Takeaway

Use inheritance for substitutable `is-a` specialisation, composition for `has-a` relationships, and avoid forcing a hierarchy when a shared interface is enough.

## Problem 36 — Refactoring a Procedural Workflow

### Problem

Refactor global standardisation state into a class and compare the designs.

### Solution

The class gives ownership of `mean_` and `std_` to each transformer instance, eliminating global interference and allowing multiple independently fitted objects.

#### Implementation

In [ ]:
class Standardizer36:
    def __init__(self):
        self.mean_ = None
        self.std_ = None

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        self.mean_ = X.mean(axis=0)
        self.std_ = X.std(axis=0)
        if np.any(self.std_ == 0):
            raise ValueError("Zero-variance feature.")
        return self

    def transform(self, X):
        if self.mean_ is None:
            raise RuntimeError("Not fitted.")
        return (np.asarray(X, dtype=float) - self.mean_) / self.std_


s36 = Standardizer36().fit(np.array([[1.0], [2.0], [3.0]]))
print(s36.transform(np.array([[4.0], [5.0]])))

### Key Takeaway

OOP is particularly useful when several operations must share persistent state without relying on global variables.

## Problem 37 — Two Independently Fitted Objects

### Problem

Fit two standardizers on different datasets and verify independent learned state.

### Solution

Each instance owns its own `mean_` and `std_`; fitting one object does not mutate the other.

#### Implementation

In [ ]:
standardizer_a37 = Standardizer().fit(
    np.array([[0.0, 10.0], [2.0, 20.0]])
)

standardizer_b37 = Standardizer().fit(
    np.array([[100.0, 1000.0], [200.0, 2000.0]])
)

print("A mean:", standardizer_a37.mean_)
print("B mean:", standardizer_b37.mean_)

### Key Takeaway

Independent instance state is essential when many fitted models or transformers coexist in one process.

## Problem 38 — Interface-Based Evaluation

### Problem

Write an evaluation function that requires only `fit()` and `predict()` and test it with two different models.

### Solution

The evaluator depends on the behavioural contract rather than the internal implementation or inheritance hierarchy.

#### Implementation

In [ ]:
def evaluate_model(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    return np.mean((y_test - predictions) ** 2)


X_train38 = np.array([[0.0], [1.0], [2.0], [3.0]])
y_train38 = np.array([0.0, 1.0, 2.0, 3.0])
X_test38 = np.array([[4.0], [5.0]])
y_test38 = np.array([4.0, 5.0])

for model in [LinearModel(), MeanModel()]:
    mse = evaluate_model(
        model,
        X_train38,
        y_train38,
        X_test38,
        y_test38
    )
    print(type(model).__name__, mse)

### Key Takeaway

Interface-oriented code is extensible because new implementations can be substituted without rewriting client logic.

## Problem 39 — Diagnose a Poor Class

### Problem

Critique a `Utilities` class containing unrelated static methods.

### Solution

The class has no coherent state or domain abstraction; it is merely a namespace for unrelated functions. A clearer design is to place statistical, I/O, messaging, and geometry functions in suitable modules or domain-specific abstractions.

### Key Takeaway

A class is not automatically better organisation than a module. OOP should represent a coherent object concept, not just group unrelated functions.

## Problem 40 — Integrated OOP Reasoning

### Problem

Analyse a transformer hierarchy and a composed pipeline containing a transformer and model.

### Solution

`Transformer` defines a conceptual interface and a default `fit()` implementation. `MeanCenterer` inherits that type relationship but overrides `fit()` and `transform()` to learn and use `mean_`. Returning `self` supports fluent estimator-style use. `Pipeline` composes components because it has a transformer and a model; it is neither of those things. Polymorphism permits substituting compatible components. Calling `predict()` before fitting causes `MeanCenterer.transform()` to raise because learned state is absent. Pipeline attributes belong to the pipeline instance; `mean_` belongs to the transformer instance and fitted model state belongs to the model instance.

#### Implementation

In [ ]:
class Transformer:
    def fit(self, X):
        return self

    def transform(self, X):
        raise NotImplementedError


class MeanCenterer(Transformer):
    def __init__(self):
        self.mean_ = None

    def fit(self, X):
        self.mean_ = np.asarray(X).mean(axis=0)
        return self

    def transform(self, X):
        if self.mean_ is None:
            raise RuntimeError("Transformer is not fitted.")
        return np.asarray(X) - self.mean_


class SimpleLinearModel:
    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        X_aug = np.column_stack([np.ones(len(X)), X])
        self.coef_ = np.linalg.pinv(X_aug) @ y
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        X_aug = np.column_stack([np.ones(len(X)), X])
        return X_aug @ self.coef_


class Pipeline40:
    def __init__(self, transformer, model):
        self.transformer = transformer
        self.model = model

    def fit(self, X, y):
        X_transformed = self.transformer.fit(X).transform(X)
        self.model.fit(X_transformed, y)
        return self

    def predict(self, X):
        X_transformed = self.transformer.transform(X)
        return self.model.predict(X_transformed)


X40 = np.array([
    [1.0, 10.0],
    [2.0, 20.0],
    [3.0, 30.0],
    [4.0, 40.0]
])
y40 = np.array([1.0, 2.0, 3.0, 4.0])

pipeline40 = Pipeline40(
    transformer=MeanCenterer(),
    model=SimpleLinearModel()
)

pipeline40.fit(X40, y40)
print("learned transformer state:", pipeline40.transformer.mean_)
print("predictions:", pipeline40.predict(X40))

### Key Takeaway

The architecture combines state, behaviour, inheritance, composition, polymorphism, and interface design in the same pattern used by many machine-learning APIs.

# Final Review

The problems in this notebook develop OOP from basic class construction to the design patterns that appear naturally in scientific and machine-learning software.

The most transferable ideas are:

- a **class** defines a reusable object abstraction, while each **instance** owns its own state;
- instance methods operate through `self`, class methods through `cls`, and static methods require neither implicit context;
- **properties** expose derived or controlled state through an attribute-like interface;
- special methods integrate custom classes with Python's data model;
- **inheritance** should represent meaningful specialisation, not merely code reuse;
- `super()` follows the method resolution order and enables cooperative inheritance;
- **polymorphism** allows client code to depend on stable behaviour rather than implementation details;
- **composition** is often preferable when one object contains or coordinates other objects;
- **dataclasses** remove repetitive machinery from data-oriented classes;
- stateful ML components naturally fit an estimator-style object lifecycle such as `fit()`, `transform()`, and `predict()`.

Good OOP does not mean expressing everything as a class. The objective is to choose an abstraction that makes state ownership, behaviour, interfaces, and relationships easier to understand and maintain.